In [2]:
# ==========================================================
# HEp-2 Image Preprocessing
# ----------------------------------------------------------
# Purpose:
# Prepare the HEp-2 dataset for ResNet50 and DenseNet121
# model development.
#
# Preprocessing steps:
# - Load the local dataset
# - Organize image paths and labels
# - Split the dataset
# - Resize images
# - Convert grayscale images to 3-channel RGB
# - Normalize images
# - Create PyTorch Dataset and DataLoaders
#
# Objective Connection:
# This preprocessing prepares the data required for
# Objective 1: Deep-learning based HEp-2 pattern recognition.
# ==========================================================

import os
import numpy as np
import pandas as pd

from PIL import Image

data_dir = os.path.join("hep2_data", "data")

class_names = sorted([
    folder
    for folder in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, folder))
])

print("Dataset path:", data_dir)
print("Classes:", class_names)

Dataset path: hep2_data\data
Classes: ['Centromere', 'Golgi', 'Homogeneous', 'NuMem', 'Nucleolar', 'Speckled']


In [3]:
# ==========================================================
# Create Image Metadata DataFrame
# ----------------------------------------------------------
# Purpose:
# Create a structured table containing the path of every
# HEp-2 image and its corresponding staining-pattern label.
#
# Why?
# This provides an organized representation of the dataset
# that can be safely divided into training, validation, and
# test sets.
# ==========================================================

image_records = []

for class_name in class_names:

    class_path = os.path.join(data_dir, class_name)

    for file_name in os.listdir(class_path):

        if file_name.lower().endswith((".png", ".jpg", ".jpeg")):

            image_records.append({
                "image_path": os.path.join(class_path, file_name),
                "label": class_name
            })

df = pd.DataFrame(image_records)

print("Total images:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts().sort_index())

Total images: 13596

Class distribution:
label
Centromere     2741
Golgi           724
Homogeneous    2494
NuMem          2208
Nucleolar      2598
Speckled       2831
Name: count, dtype: int64


In [5]:
# ==========================================================
# Install Scikit-learn
# ----------------------------------------------------------
# Purpose:
# Install Scikit-learn for dataset splitting and later
# model evaluation.
# ==========================================================

!pip install -q scikit-learn

print("Scikit-learn installed successfully.")

Scikit-learn installed successfully.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# ==========================================================
# Stratified Train / Validation / Test Split
# ----------------------------------------------------------
# Purpose:
# Divide the HEp-2 dataset into training, validation, and
# test sets while preserving the class distribution.
#
# Split:
# - 70% Training
# - 15% Validation
# - 15% Testing
#
# Why?
# Stratification prevents the class imbalance from becoming
# worse in any individual split and provides a fair basis
# for model evaluation.
#
# Objective Connection:
# This creates reliable data partitions for Objective 1
# (model development) and Objective 3 (model comparison).
# ==========================================================

from sklearn.model_selection import train_test_split

# First split: 70% training, 30% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

# Second split: divide the temporary 30% equally
# into validation (15%) and test (15%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Dataset split completed.\n")

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTesting distribution:")
print(test_df["label"].value_counts().sort_index())

Dataset split completed.

Training images: 9517
Validation images: 2039
Testing images: 2040

Training distribution:
label
Centromere     1919
Golgi           507
Homogeneous    1746
NuMem          1545
Nucleolar      1818
Speckled       1982
Name: count, dtype: int64

Validation distribution:
label
Centromere     411
Golgi          109
Homogeneous    374
NuMem          331
Nucleolar      390
Speckled       424
Name: count, dtype: int64

Testing distribution:
label
Centromere     411
Golgi          108
Homogeneous    374
NuMem          332
Nucleolar      390
Speckled       425
Name: count, dtype: int64


In [7]:
# ==========================================================
# Define Image Preprocessing Transformations
# ----------------------------------------------------------
# Purpose:
# Define the transformations required to prepare HEp-2
# images for the CNN models.
#
# Steps:
# - Resize 78 × 78 images to 224 × 224
# - Convert grayscale images to 3-channel RGB
# - Normalize pixel values using ImageNet statistics
#
# Why?
# ResNet50 and DenseNet121 use 224 × 224 RGB input and
# ImageNet-pretrained weights.
# ==========================================================

import torch
from torchvision import transforms

IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Image transformations defined successfully.")
print("CNN input size:", IMAGE_SIZE, "x", IMAGE_SIZE)

Image transformations defined successfully.
CNN input size: 224 x 224


In [8]:
# ==========================================================
# Create PyTorch Dataset Class
# ----------------------------------------------------------
# Purpose:
# Create a custom PyTorch Dataset to load HEp-2 images
# together with their corresponding class labels.
#
# Why?
# The Dataset provides images to the CNN models in a
# consistent format while applying the appropriate
# preprocessing transformations.
# ==========================================================

from torch.utils.data import Dataset

# Create numerical labels for the six HEp-2 classes
label_to_index = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

index_to_label = {
    index: class_name
    for class_name, index in label_to_index.items()
}

print("Label mapping:")
for index, class_name in index_to_label.items():
    print(f"{index}: {class_name}")


class HEP2Dataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        image_path = self.dataframe.loc[index, "image_path"]
        label_name = self.dataframe.loc[index, "label"]

        image = Image.open(image_path).convert("L")

        # Convert grayscale image to RGB before applying transforms
        image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = label_to_index[label_name]

        return image, label


print("\nPyTorch Dataset class created successfully.")

Label mapping:
0: Centromere
1: Golgi
2: Homogeneous
3: NuMem
4: Nucleolar
5: Speckled

PyTorch Dataset class created successfully.


In [9]:
# ==========================================================
# Create Dataset Objects
# ----------------------------------------------------------
# Purpose:
# Create separate PyTorch Dataset objects for the training,
# validation, and test sets.
#
# Why?
# Each split requires different transformations:
# - Training: preprocessing + data augmentation
# - Validation: preprocessing only
# - Testing: preprocessing only
#
# This ensures that augmentation is applied only to training
# data and does not affect model evaluation.
# ==========================================================

train_dataset = HEP2Dataset(
    train_df,
    transform=train_transform
)

val_dataset = HEP2Dataset(
    val_df,
    transform=val_test_transform
)

test_dataset = HEP2Dataset(
    test_df,
    transform=val_test_transform
)

print("PyTorch Dataset objects created successfully.\n")

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

PyTorch Dataset objects created successfully.

Training samples: 9517
Validation samples: 2039
Testing samples: 2040


In [10]:
# ==========================================================
# Create PyTorch DataLoaders
# ----------------------------------------------------------
# Purpose:
# Create DataLoaders to efficiently load batches of HEp-2
# images during model training and evaluation.
#
# Why?
# Batch-based loading makes training more efficient and
# allows the CNN models to process the dataset systematically.
# ==========================================================

from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created successfully.\n")

print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

DataLoaders created successfully.

Batch size: 32
Training batches: 298
Validation batches: 64
Testing batches: 64


In [11]:
# ==========================================================
# Verify DataLoader Output
# ----------------------------------------------------------
# Purpose:
# Verify that images and labels are loaded correctly and
# have the expected dimensions before model training.
# ==========================================================

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Image data type: torch.float32
Label data type: torch.int64
